# Exploring the YouTube Data API

**Project:** YouTube Title Predictor (ML Zoomcamp midterm)
**Purpose:** Testing the API in a notebook lets me check if it is returning the information I need and that its in the right fromat.

This is a scratch notebook for learning how the API responds before I build
the real data collection script. The graded analysis lives in `notebook.ipynb`.

## What's in here
1. Load the API key from `.env` (never printed)
2. The call asks for 3 parts: `snippet`, `contentDetails` and `statistics`
3. Pulled out from the json response was the `Channel ID`, `Channel title`, `Subscriber count`, `Video count` and ` Uploads playlist`. This will later be used to create a dataset. 

## Notes / findings
- `subscriberCount` comes back as strings, so they need to be converted to numbers before doing any analysis.
- The upload playlist ID is the channel ID with the second letter changed from `C` to `U` (`UCqxY...`  → `UUqxY...`); everything after that is identical.  Google doesn't officially document this, so `collect_data.py` will read the ID from `contentDetails.relatedPlaylists.uploads` rather than build it. The pattern is still a handy sanity check.
- This call costs 1 quota unit (out of 10,000 a day).
- only 3 of the seed channels fall within the 1k-100k subscribers, well-known channels tend to be too big. The list will be topped up with `search.list`.
- `search.list` only supports `part="snippet"`, so it returns no subscriber
  counts. A follow-up `channels.list` call is needed to get those.
- In search results, `id` is a dict, not a string: the channel ID is at
  `item["id"]["channelId"]` (in `channels.list` it's just `item["id"]`).
- Each `search.list` call costs 100 quota units and returns at most 50 results.
  The response includes a `nextPageToken` when more results exist, which can
  be sent back to get the next page (another 100 units).
- Test query "couple travel Southeast Asia" returned 50 channels; the first few
  (e.g. Jaime and Adam, Mai and Chris Travel, Cam & Evie Abroad) look like
  close matches to KarmicSteps.



In [12]:
import os
from dotenv import load_dotenv
from requests import request
import pandas as pd

## 1. Load the api_key

In [2]:
load_dotenv()
api_key = os.getenv("YOUTUBE_API_KEY")
print("api_key is None:", api_key is None)
print("api_key length:", len(api_key))


api_key is None: False
api_key length: 39


## 2. Call channels.list

In [3]:
URL = "https://www.googleapis.com/youtube/v3/channels"

params = {
    "part": "snippet,contentDetails,statistics",
    "forHandle": "@KarmicSteps",
    "key": api_key,
}

# send a GET request to URL
response = get(URL, params=params, timeout=10)

print("Response status code:", response.status_code)

Response status code: 200


## 3. Explore the JSON

In [4]:
# convert the response from JSON into a Python dict
print("Response status code:", response.status_code)
data = response.json()
print("Top-level keys of data:", data.keys())
if "items" in data:
    print("Top-level keys of first item:", data["items"][0].keys())

Response status code: 200
Top-level keys of data: dict_keys(['kind', 'etag', 'pageInfo', 'items'])
Top-level keys of first item: dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics'])


## 4. Pull out the channel data

In [5]:
# Print channel data
channel = data["items"][0]
print("Channel ID:", channel["id"])
print("Channel title:", channel["snippet"]["title"])
print("Subscriber count:", channel["statistics"]["subscriberCount"])
print("Video count:", channel["statistics"]["videoCount"])
print("Uploads playlist ID:", channel["contentDetails"]["relatedPlaylists"]["uploads"])

Channel ID: UCqxY_VZtLNg036hIA-wRahQ
Channel title: Karmic Steps
Subscriber count: 343
Video count: 100
Uploads playlist ID: UUqxY_VZtLNg036hIA-wRahQ


## 5. Check seed channels 

In [ ]:
# define a function get_channel(handle)
def get_channel(handle):
    params = {
        "part": "snippet,statistics,contentDetails",
        "forHandle": handle,
        "key": api_key,
    }
    response = get(URL, params=params, timeout=10)
    
    print("Response status code:", response.status_code)

    data = response.json()
    if "items" not in data:
        return None
    item = data["items"][0]
    return {
        "handle": handle,
        "channel_id": item["id"],
        "title": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"]
    }

seeds = pd.read_csv("data/channels_seed.csv")

rows = []
for handle in seeds['handle']:
    results = get_channel(handle)
    if results is None:
        print(f"Warning: handle {handle} not found")
    else:
        rows.append(results)

df = pd.DataFrame(rows)
df = df.sort_values(by='subscribers', ascending=False)
df 

In [20]:
# Filter the seed table
df[(df["subscribers"] >= 1000) & (df["subscribers"] <= 100000)]

,handle,channel_id,title,subscribers,video_count,uploads_playlist_id
3,@AcrossEveryBorder,UCYAIbGWcLdohnpRh-YLXzRw,Sara & Ryan,16900,137,UUYAIbGWcLdohnpRh-YLXzRw
1,@arjayneyra,UCGloEV_XBzi-3Fp2hSnKVWg,Arjay Neyra,4840,466,UUGloEV_XBzi-3Fp2hSnKVWg
8,@TannerandSierra,UCf9kneMiWCVgc47Wzi3ew6A,Tanner and Sierra,4790,600,UUf9kneMiWCVgc47Wzi3ew6A


## 6. Test Search List

Using the API method `search.list` to search for more channels within the niche. However search list does not return subscriber count. So a second API method `channels.list` and channel ID will be used to retrieve the subscribers, video count and upload playlist.

Search phrases: 
- couple travel vlog
- budget travel tips
- backpacking through South East Asia
- Living in Thailand
- Living in Korea
- couple travel Southeast Asia
- Van life in Southern Africa

In [21]:
# store the search endpoint URL in a variable called SEARCH_URL
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"

params = {
    "part": "snippet",
    "q": "couple travel Southeast Asia",
    "type": "channel",
    "maxResults": 50,
    "relevanceLanguage": 'en',
    "key": api_key,
}

response = get(SEARCH_URL, params=params, timeout=10)

print("Response status code:", response.status_code)


data = response.json()


Response status code: 200


In [24]:
print(data.keys())
print(len(data["items"]))
print(data["items"][0].keys())
print(data["items"][0]["id"])

dict_keys(['kind', 'etag', 'nextPageToken', 'regionCode', 'pageInfo', 'items'])
50
dict_keys(['kind', 'etag', 'id', 'snippet'])
{'kind': 'youtube#channel', 'channelId': 'UClhQmYpyIedwL-gAW62nF_Q'}


In [35]:
for item in data["items"][:20]:
    print(item["snippet"]["title"])


Warick and Aireen - Island Life
Jaime and Adam
Mai and Chris Travel
Cam & Evie Abroad
TravelGlitchx30
Rhys & Jade
SS Travels | Southeast Asia Guides
Carla and Poppy Travel
MEonthemap | Travel Couple
Joss and Megan Travel 
Surbhi & Chetan
Avoir Travel
Emma & Tyler
SamAndRachTravel
WanderOH - Travel couple Olaf & Hanna
Age Gap No Map
Billie and Matt Travel
Tom And Lauren Travel
Jordan and Matt
Twogether Nomads


In [32]:
first = data["items"][0]
print(first["id"])

print(first["id"]["channelId"])

{'kind': 'youtube#channel', 'channelId': 'UClhQmYpyIedwL-gAW62nF_Q'}
UClhQmYpyIedwL-gAW62nF_Q


In [34]:
channel_ids = []

for i in data["items"]:
        channel_ids.append(i["id"]["channelId"])



print(len(channel_ids))
print(channel_ids[:3])

50
['UClhQmYpyIedwL-gAW62nF_Q', 'UCLk1JGa2AtMqz1fAVFE20Rw', 'UCD9CjkcRJdP_QrQUb7p1srg']
